# 03 · Generación y Carga de Datos Sintéticos (con Faker)

Genera datos realistas en **múltiples locales** (`es_ES`, `es_MX`, `en_US`, `pt_BR`) y los sube a BigQuery.

| Tabla | Filas aprox. |
|---|---|
| categories | 10 |
| customers | 500 |
| products | 150 |
| orders | 2 000 |
| order_items | ~5 500 |
| payments | ~1 900 |
| reviews | ~1 000 |

In [1]:
# Si no tienes Faker instalado, descomenta:
# !pip install faker

# Configuración Inicial 

In [1]:
import os
import random
from datetime import datetime, timedelta, timezone

import pandas as pd
from dotenv import load_dotenv
from faker import Faker
from google.cloud import bigquery

load_dotenv()
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
PROJECT_ID = os.getenv('GCP_PROJECT_ID')
DATASET_ID = os.getenv('BQ_DATASET_ID')
client = bigquery.Client(project=PROJECT_ID)

LOCALES = ['es_ES', 'es_MX', 'en_US', 'pt_BR']
fake = Faker(LOCALES)
Faker.seed(42)
random.seed(42)

# Un Faker por locale para acceder a metodos como current_country()
fakers = {loc: Faker(loc) for loc in LOCALES}
for f in fakers.values():
    f.seed_instance(42)

START = datetime(2023, 1, 1, tzinfo=timezone.utc)
END   = datetime(2026, 4, 30, tzinfo=timezone.utc)

def rand_ts(a, b):
    delta = b - a
    return a + timedelta(seconds=random.randint(0, int(delta.total_seconds())))

print('Entorno listo ✓')
print(f'Ejemplo: {fake.name()} | {fake.email()} | {fake.city()}')

Entorno listo ✓
Ejemplo: Raimundo Llopis Hierro | felixtrejo@example.org | Alves


# ── 1. categories (datos de negocio fijos) ─────────────────────

In [ ]:
# ── 1. categories (datos de negocio fijos) ─────────────────────-
CATEGORY_DATA = [
    ('Electrónica',      'Gadgets, teléfonos, portátiles y accesorios'),
    ('Ropa',             'Ropa para hombre y mujer'),
    ('Hogar y Jardín',   'Muebles, decoración y herramientas de jardín'),
    ('Deportes',         'Equipos de fitness y artículos para actividades al aire libre'),
    ('Libros',           'Ficción, no ficción y libros de texto'),
    ('Juguetes',         'Juguetes y juegos para niños'),
    ('Belleza',          'Cuidado de la piel, maquillaje y cuidado personal'),
    ('Automotriz',       'Repuestos y accesorios para coches'),
    ('Alimentos y Comestibles', 'Alimentos envasados y bebidas'),
    ('Salud',            'Vitaminas, suplementos y dispositivos médicos'),
]
df_categories = pd.DataFrame([
    {'category_id': i + 1, 'name': n, 'description': d}
    for i, (n, d) in enumerate(CATEGORY_DATA)
])
print(f'categories: {len(df_categories)} filas ✓')
df_categories

categories: 10 filas ✓


,category_id,name,description
0,1,Electrónica,"Gadgets, teléfonos, portátiles y accesorios"
1,2,Ropa,Ropa para hombre y mujer
2,3,Hogar y Jardín,"Muebles, decoración y herramientas de jardín"
3,4,Deportes,Equipos de fitness y artículos para actividade...
4,5,Libros,"Ficción, no ficción y libros de texto"
5,6,Juguetes,Juguetes y juegos para niños
6,7,Belleza,"Cuidado de la piel, maquillaje y cuidado personal"
7,8,Automotriz,Repuestos y accesorios para coches
8,9,Alimentos y Comestibles,Alimentos envasados y bebidas
9,10,Salud,"Vitaminas, suplementos y dispositivos médicos"


# ── 2. customers  (Faker: nombres, emails, paises, ciudades) ───

In [ ]:
# ── 2. customers  (Faker: nombres, emails, paises, ciudades) ───
CHANNELS = ['orgánico', 'búsqueda_pagada', 'redes_sociales', 'referencia', 'correo_electrónico', 'directo']

rows = []
for i in range(500):
    loc   = random.choice(LOCALES)
    f_loc = fakers[loc]
    rows.append({
        'customer_id':         i + 1,
        'name':                f_loc.name(),
        'email':               fake.unique.email(),
        'country':             f_loc.current_country(),
        'city':                f_loc.city(),
        'acquisition_channel': random.choice(CHANNELS),
        'registered_at':       rand_ts(START - timedelta(days=365), END),
    })

df_customers = pd.DataFrame(rows)
print(f'customers: {len(df_customers)} filas ✓')
df_customers.head(5)

# ── 3. products  (nombres base + variante de color/version) ────

In [ ]:
# ── 3. products  (nombres base + variante de color/version) ────
PRODUCT_TEMPLATES = [
    ('Auriculares Inalámbricos Pro', 1, 89.99, 35.0), ('Funda para Smartphone X12', 1, 19.99, 4.5),
    ('Hub USB-C 7 en 1', 1, 45.00, 16.0),           ('Monitor LED 27"', 1, 229.00, 95.0),
    ('Teclado Mecánico', 1, 79.99, 28.0),            ('Zapatillas de Running', 2, 69.99, 22.0),
    ('Chaqueta Vaquera', 2, 59.99, 18.0),            ('Vestido de Verano', 2, 39.99, 12.0),
    ('Polo', 2, 29.99, 8.0),                         ('Jersey de Lana', 2, 54.99, 17.0),
    ('Mesa de Centro', 3, 149.00, 55.0),             ('Manguera de Jardín 30m', 3, 34.99, 11.0),
    ('Set de Macetas', 3, 24.99, 7.5),               ('Lámpara de Escritorio LED', 3, 39.99, 13.0),
    ('Esterilla de Yoga Premium', 4, 29.99, 9.0),    ('Set de Mancuernas 20kg', 4, 89.99, 32.0),
    ('Casco de Ciclismo', 4, 49.99, 17.0),           ('Recetario de Python', 5, 44.99, 10.0),
    ('Manual de Ciencia de Datos', 5, 39.99, 9.0),   ('Novela: El Último Código', 5, 14.99, 3.0),
    ('Set de LEGO City', 6, 59.99, 20.0),            ('Juego de Mesa Catan', 6, 39.99, 13.0),
    ('Crema Hidratante SPF50', 7, 24.99, 7.0),       ('Perfume EDP 50ml', 7, 79.99, 22.0),
    ('Soporte de Coche para Móvil', 8, 14.99, 4.0),  ('Cámara de Salpicadero 4K', 8, 119.00, 42.0),
    ('Granola Orgánica 500g', 9, 9.99, 3.5),         ('Caja de Té Verde 100u', 9, 7.99, 2.5),
    ('Vitamina D3 1000IU', 10, 12.99, 4.0),          ('Tensiómetro de Brazo', 10, 59.99, 21.0),
]

COLORS = ['Rojo', 'Azul', 'Negro', 'Blanco', 'Verde', 'Gris', 'Azul Marino', 'Beige']

rows = []
for i in range(150):
    name, cat_id, bp, bc = PRODUCT_TEMPLATES[i % len(PRODUCT_TEMPLATES)]
    if i >= len(PRODUCT_TEMPLATES):
        # Variante con color para categorias de moda/tech, version para el resto
        suffix = f' - {random.choice(COLORS)}' if cat_id in (1,2,4,7) else f' v{i//len(PRODUCT_TEMPLATES)+1}'
        name   = name + suffix
    rows.append({
        'product_id':  i + 1,
        'category_id': cat_id,
        'name':        name,
        'price':       round(bp * random.uniform(0.85, 1.15), 2),
        'cost':        round(bc * random.uniform(0.90, 1.10), 2),
        'stock':       fake.random_int(min=0, max=500),
        'is_active':   fake.boolean(chance_of_getting_true=95),
    })

df_products = pd.DataFrame(rows)
print(f'products: {len(df_products)} filas ✓')
df_products.head(5)

# ── 4. orders + 5. order_items ─────────────────────────────────

In [13]:
# ── 4. orders + 5. order_items ─────────────────────────────────
ORDER_STATUSES = ['pendiente', 'en_proceso', 'enviado', 'entregado', 'cancelado', 'devuelto']
STATUS_WEIGHTS = [0.05, 0.10, 0.15, 0.60, 0.07, 0.03]

orders_rows = []
items_rows  = []
item_id     = 1

for oid in range(1, 2001):
    cust    = df_customers.iloc[random.randint(0, 499)]
    ordered = rand_ts(START, END)
    status  = random.choices(ORDER_STATUSES, STATUS_WEIGHTS)[0]

    shipped = delivered = None
    if status in ('enviado', 'entregado', 'devuelto'):
        shipped = ordered + timedelta(days=random.randint(1, 4))
    if status in ('entregado', 'devuelto'):
        delivered = shipped + timedelta(days=random.randint(2, 10))

    # Dirección generada con Faker (locale variado)
    f_loc   = fakers[random.choice(LOCALES)]
    address = f"{f_loc.street_address()}, {f_loc.city()}, {f_loc.current_country()}"

    orders_rows.append({
        'order_id':         oid,
        'customer_id':      int(cust['customer_id']),
        'status':           status,
        'shipping_address': address,
        'ordered_at':       ordered,
        'shipped_at':       shipped,
        'delivered_at':     delivered,
    })

    for pid in random.sample(range(1, 151), random.randint(1, 5)):
        prod = df_products[df_products['product_id'] == pid].iloc[0]
        items_rows.append({
            'order_item_id': item_id,
            'order_id':      oid,
            'product_id':    pid,
            'quantity':      fake.random_int(min=1, max=4),
            'unit_price':    float(prod['price']),
            'discount':      random.choice([0, 0, 0, 0.05, 0.10, 0.15, 0.20]),
        })
        item_id += 1

df_orders = pd.DataFrame(orders_rows)
df_items  = pd.DataFrame(items_rows)
print(f'orders: {len(df_orders)} | order_items: {len(df_items)} ✓')
df_orders.head(3)

In [ ]:
# ── 6. payments ────────────────────────────────────────────────
METHODS = ['tarjeta_de_crédito', 'tarjeta_de_débito', 'paypal', 'transferencia_bancaria', 'criptomonedas']
PAY_ST  = ['completado', 'pendiente', 'fallido', 'reembolsado']
PAY_W   = [0.80, 0.10, 0.05, 0.05]

payments_rows = []
for _, row in df_orders.iterrows():
    if row['status'] == 'cancelled' and random.random() > 0.30:
        continue
    items  = df_items[df_items['order_id'] == row['order_id']]
    amount = round(
        sum(r['unit_price'] * r['quantity'] * (1 - r['discount']) for _, r in items.iterrows()), 2
    )
    paid_at = (
        row['ordered_at'] + timedelta(minutes=fake.random_int(min=1, max=60))
        if row['ordered_at'] and random.random() > 0.02 else None
    )
    payments_rows.append({
        'payment_id': len(payments_rows) + 1,
        'order_id':   int(row['order_id']),
        'method':     random.choice(METHODS),
        'status':     random.choices(PAY_ST, PAY_W)[0],
        'amount':     amount,
        'paid_at':    paid_at,
    })

df_payments = pd.DataFrame(payments_rows)
print(f'payments: {len(df_payments)} filas ✓')
df_payments.head(3)

# ── 7. reviews  (comentarios coherentes con el rating) ─────────

In [ ]:
# ── 7. reviews  (comentarios coherentes con el rating) ─────────
POSITIVE = [
    '¡Producto excelente, muy satisfecho!', 'Funciona exactamente como se describe.',
    'Buena relación calidad-precio.', 'Entrega rápida y calidad excelente.',
    'Lo recomendaría sin dudarlo.', '¡Superó mis expectativas!',
    'Volvería a comprarlo sin dudarlo.', '¡Perfecto, cinco estrellas! ',
]
NEGATIVE = [
    'No era lo que esperaba.', 'Producto mediocre, nada especial.',
    'Mal embalaje pero el artículo está bien.', 'Llegó tarde y algo dañado.',
    'No lo recomendaría.',
]

delivered_oids = set(df_orders[df_orders['status'] == 'entregado']['order_id'])
reviewable     = df_items[df_items['order_id'].isin(delivered_oids)]

reviews_rows = []
for _, item in reviewable.iterrows():
    if random.random() > 0.40:
        continue
    order_row = df_orders[df_orders['order_id'] == item['order_id']].iloc[0]
    delivered = order_row['delivered_at']
    reviewed  = delivered + timedelta(days=fake.random_int(min=1, max=30)) if delivered else None
    rating    = random.choices([1,2,3,4,5], weights=[0.05,0.08,0.12,0.30,0.45])[0]
    if random.random() < 0.15:
        comment = None
    elif rating >= 4:
        comment = random.choice(POSITIVE)
    else:
        comment = random.choice(NEGATIVE)

    reviews_rows.append({
        'review_id':     len(reviews_rows) + 1,
        'order_item_id': int(item['order_item_id']),
        'rating':        rating,
        'comment':       comment,
        'reviewed_at':   reviewed,
    })

df_reviews = pd.DataFrame(reviews_rows)
print(f'reviews: {len(df_reviews)} filas ✓')
df_reviews.head(5)

# ── Resumen antes de subir ─────────────────────────────────────


In [ ]:
# ── Resumen antes de subir ─────────────────────────────────────
for nombre, df in [('categories',df_categories),('customers',df_customers),
                   ('products',df_products),('orders',df_orders),
                   ('order_items',df_items),('payments',df_payments),('reviews',df_reviews)]:
    print(f'  {nombre:<20} {len(df):>6} filas')

# ── Carga a BigQuery (WRITE_TRUNCATE = sobreescribe si ya existe) ─

In [ ]:
# ── Carga a BigQuery (WRITE_TRUNCATE = sobreescribe si ya existe) ─
TABLE_MAP = {
    'categories':    df_categories,
    'customers':     df_customers,
    'products':      df_products,
    'orders':        df_orders,
    'order_item_id': df_items,
    'payments':      df_payments,
    'reviews':       df_reviews,
}
cfg = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')

for tname, df in TABLE_MAP.items():
    ref = f'{PROJECT_ID}.{DATASET_ID}.{tname}'
    job = client.load_table_from_dataframe(df, ref, job_config=cfg)
    job.result()
    print(f'  ✓ {tname:<22} {len(df):>6} filas cargadas')

print('\n🎉 ¡Carga completada!')